# Phase 2, Workshop 02 — Cell Content: Colour Bars, Icons, and Prices

The empty grid from Workshop 01 shows where each space is, but it does not look like a Monopoly board yet. In this workshop you will add the distinctive colour strips that identify property groups, icons for special squares, and price labels. By the end, your board will be instantly recognisable.

**What you will learn:**
- Absolute positioning for colour bars inside grid cells
- How the colour bar position changes depending on which side of the board the cell is on
- Building cell HTML dynamically from the board data array
- Using emoji as lightweight icons

**Time:** roughly 15 to 20 minutes.

## Section 1 — Property Colour Bars

On a real Monopoly board, each property has a coloured strip along one edge. The strip sits on the *inside* edge of the board (the edge closest to the centre):

- **Bottom row**: colour bar at the **top** of the cell
- **Left column**: colour bar on the **right** of the cell
- **Top row**: colour bar at the **bottom** of the cell
- **Right column**: colour bar on the **left** of the cell

We achieve this with **absolute positioning**. The colour bar is a small `<div>` inside the cell, positioned using `position: absolute` with the appropriate edge set to 0.

## Section 2 — The Updated Board Data

We need to expand our board data to include colour codes, costs, and icons. Here is the compact format we will use:

```javascript
{ n: "Old Kent Rd", t: "prop", c: 60, r: 10, col: "var(--brown)" }
{ n: "Chance",      t: "chance", icon: "❓" }
{ n: "Income Tax",  t: "tax",   a: 200, icon: "💸" }
```

Each object has:
- `n` = name, `t` = type
- `c` = cost, `r` = rent (for properties)
- `col` = CSS colour (for the colour bar)
- `icon` = an emoji (for special squares)
- `a` = amount (for taxes)

## Section 3 — Build the Board with Colour Bars

In [ ]:
import os

project_folder = os.path.join(os.path.expanduser("~"), "text_monopoly", "phase2")
os.makedirs(project_folder, exist_ok=True)

html_content = """<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Phase 2 - Colour Bars and Icons</title>
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Outfit:wght@400;600;700;900&display=swap');

        :root {
            --bg: #1a1520;
            --board-bg: #f5f0e8;
            --board-border: #1a1520;
            --board-center: #dcd4c8;
            --text-dark: #1a1520;
            --brown: #8B4513; --sky: #87CEEB; --pink: #d63384;
            --orange: #fd7e14; --red: #e84855; --yellow: #ffc107;
            --dgreen: #198754; --navy: #0d6efd; --station: #444; --utility: #aaa;
        }

        * { margin: 0; padding: 0; box-sizing: border-box; }
        body {
            font-family: 'Outfit', sans-serif;
            background: var(--bg);
            display: flex; justify-content: center; padding: 30px;
        }

        .board {
            display: grid;
            grid-template-columns: 72px repeat(9, 56px) 72px;
            grid-template-rows: 72px repeat(9, 56px) 72px;
            gap: 1px;
            background: var(--board-border);
            border: 3px solid var(--board-border);
            border-radius: 8px;
            overflow: hidden;
            font-size: 0.5rem;
        }

        .cell {
            background: var(--board-bg);
            color: var(--text-dark);
            display: flex;
            flex-direction: column;
            align-items: center;
            justify-content: center;
            position: relative;     /* needed for absolute children */
            padding: 2px;
            text-align: center;
            line-height: 1.15;
            font-weight: 600;
            overflow: hidden;
        }

        /* The colour bar */
        .cell .color-bar {
            position: absolute;
            width: 100%;
            height: 8px;
            left: 0;
        }

        /* Position depends on which side of the board */
        .side-top .color-bar    { bottom: 0; }
        .side-bottom .color-bar { top: 0; }
        .side-left .color-bar   {
            top: 0; left: unset; right: 0;
            width: 8px; height: 100%;  /* vertical strip */
        }
        .side-right .color-bar  {
            top: 0; left: 0;
            width: 8px; height: 100%;  /* vertical strip */
        }

        .cell .name  { font-size: 0.45rem; font-weight: 700; z-index: 1; }
        .cell .price { font-size: 0.4rem; color: #666; z-index: 1; }
        .cell .icon  { font-size: 0.9rem; z-index: 1; }

        .corner { font-size: 0.55rem; font-weight: 900; }

        .board-center {
            grid-column: 2 / 11; grid-row: 2 / 11;
            background: var(--board-center);
            display: flex; flex-direction: column;
            align-items: center; justify-content: center;
            font-size: 1.8rem; font-weight: 900;
            color: var(--text-dark); letter-spacing: -1px;
        }
    </style>
</head>
<body>
    <div class="board" id="board"></div>

    <script>
        const B = [
            { n:"GO",           t:"go",    icon:"🏁" },
            { n:"Old Kent Rd",  t:"prop",  c:60,  r:10,  col:"var(--brown)" },
            { n:"Chest",        t:"chest", icon:"📦" },
            { n:"Whitechapel",  t:"prop",  c:60,  r:10,  col:"var(--brown)" },
            { n:"Income Tax",   t:"tax",   a:200, icon:"💸" },
            { n:"Kings Cross",  t:"prop",  c:200, r:50,  col:"var(--station)" },
            { n:"Angel",        t:"prop",  c:100, r:20,  col:"var(--sky)" },
            { n:"Chance",       t:"chance", icon:"❓" },
            { n:"Euston Rd",    t:"prop",  c:100, r:20,  col:"var(--sky)" },
            { n:"Pentonville",  t:"prop",  c:120, r:25,  col:"var(--sky)" },
            { n:"Visiting",     t:"visit", icon:"👀" },
            { n:"Pall Mall",    t:"prop",  c:140, r:30,  col:"var(--pink)" },
            { n:"Electric Co",  t:"prop",  c:150, r:35,  col:"var(--utility)" },
            { n:"Whitehall",    t:"prop",  c:140, r:30,  col:"var(--pink)" },
            { n:"Northumber.",  t:"prop",  c:160, r:35,  col:"var(--pink)" },
            { n:"Marylebone",   t:"prop",  c:200, r:50,  col:"var(--station)" },
            { n:"Bow St",       t:"prop",  c:180, r:40,  col:"var(--orange)" },
            { n:"Chest",        t:"chest", icon:"📦" },
            { n:"Marlborough",  t:"prop",  c:180, r:40,  col:"var(--orange)" },
            { n:"Vine St",      t:"prop",  c:200, r:45,  col:"var(--orange)" },
            { n:"Free Parking", t:"free",  icon:"🅿️" },
            { n:"Strand",       t:"prop",  c:220, r:50,  col:"var(--red)" },
            { n:"Chance",       t:"chance", icon:"❓" },
            { n:"Fleet St",     t:"prop",  c:220, r:50,  col:"var(--red)" },
            { n:"Trafalgar Sq", t:"prop",  c:240, r:55,  col:"var(--red)" },
            { n:"Fenchurch",    t:"prop",  c:200, r:50,  col:"var(--station)" },
            { n:"Leicester Sq", t:"prop",  c:260, r:60,  col:"var(--yellow)" },
            { n:"Coventry St",  t:"prop",  c:260, r:60,  col:"var(--yellow)" },
            { n:"Water Works",  t:"prop",  c:150, r:35,  col:"var(--utility)" },
            { n:"Piccadilly",   t:"prop",  c:280, r:65,  col:"var(--yellow)" },
            { n:"GO TO JAIL",   t:"jail",  icon:"🚔" },
            { n:"Regent St",    t:"prop",  c:300, r:70,  col:"var(--dgreen)" },
            { n:"Oxford St",    t:"prop",  c:300, r:70,  col:"var(--dgreen)" },
            { n:"Chest",        t:"chest", icon:"📦" },
            { n:"Bond St",      t:"prop",  c:320, r:75,  col:"var(--dgreen)" },
            { n:"Liverpool St", t:"prop",  c:200, r:50,  col:"var(--station)" },
            { n:"Chance",       t:"chance", icon:"❓" },
            { n:"Park Lane",    t:"prop",  c:350, r:85,  col:"var(--navy)" },
            { n:"Super Tax",    t:"tax",   a:100, icon:"💸" },
            { n:"Mayfair",      t:"prop",  c:400, r:100, col:"var(--navy)" }
        ];

        function boardToGrid(i) {
            if (i <= 10) return { row: 10, col: 10 - i };
            if (i <= 20) return { row: 10 - (i - 10), col: 0 };
            if (i <= 30) return { row: 0, col: i - 20 };
            return { row: i - 30, col: 10 };
        }

        function getSide(i) {
            if (i <= 10) return "side-bottom";
            if (i <= 20) return "side-left";
            if (i <= 30) return "side-top";
            return "side-right";
        }

        const board = document.getElementById("board");

        for (let i = 0; i < 40; i++) {
            const g = boardToGrid(i);
            const sp = B[i];
            const side = getSide(i);
            const isCorner = [0, 10, 20, 30].includes(i);

            const div = document.createElement("div");
            div.className = "cell " + side + (isCorner ? " corner" : "");
            div.style.gridRow = g.row + 1;
            div.style.gridColumn = g.col + 1;

            let html = "";

            // Colour bar for properties
            if (sp.t === "prop" && sp.col) {
                html += '<div class="color-bar" style="background:' + sp.col + '"></div>';
            }

            // Icon for special squares
            if (sp.icon) {
                html += '<div class="icon">' + sp.icon + '</div>';
            }

            // Name
            html += '<div class="name">' + sp.n + '</div>';

            // Price
            if (sp.c) {
                html += '<div class="price">£' + sp.c + '</div>';
            }

            div.innerHTML = html;
            board.appendChild(div);
        }

        // Centre
        const center = document.createElement("div");
        center.className = "board-center";
        center.textContent = "MONOPOLY";
        board.appendChild(center);
    </script>
</body>
</html>
"""

filepath = os.path.join(project_folder, "p2_02_colour_bars.html")
with open(filepath, "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"Colour bar board saved to: {filepath}")
print("Open it to see the full board with colour strips, icons, and prices!")

## Section 4 — How Colour Bars Work on Different Sides

The clever part is how the same `.color-bar` element looks different depending on which side of the board the cell is on.

For **top and bottom** rows, the bar is a full-width horizontal strip:
```css
.side-bottom .color-bar { top: 0; }       /* strip at the top of the cell */
.side-top .color-bar    { bottom: 0; }    /* strip at the bottom of the cell */
```

For **left and right** columns, the bar becomes a tall narrow vertical strip:
```css
.side-left .color-bar  { right: 0; width: 8px; height: 100%; }
.side-right .color-bar { left: 0;  width: 8px; height: 100%; }
```

The default `.color-bar` style sets `width: 100%` and `height: 8px`, which works for horizontal bars. The side-specific rules override these for vertical bars.

## Section 5 — Using `position: relative` and `position: absolute`

The colour bar uses **absolute positioning** inside its parent cell. For this to work, the parent must have `position: relative`. Here is how it fits together:

```css
.cell {
    position: relative;   /* establishes positioning context */
}

.cell .color-bar {
    position: absolute;   /* positioned relative to .cell */
    top: 0;               /* flush against the top edge */
    left: 0;              /* flush against the left edge */
}
```

Without `position: relative` on the parent, the absolute child would position itself relative to the entire page instead.

## Section 6 — Challenge

1. Change the Chance icon from ❓ to 🃏 and see how it looks on the board.
2. Add a subtle hover effect to the cells: when you hover over a property, make its background slightly darker (hint: `.cell:hover { background: #ebe3d5; }`).
3. The Community Chest and Chance squares currently show the same size as properties. Try making them display only the icon (no name) when the cell is small.

In **Workshop 03** you will add player tokens to the board and learn how to move them as the game state changes. 🎯